In [1]:
# ============================================================
# LangChain create_agent with:
# - Tool calls
# - Tool-call metrics
# - Token metrics
# - Agent iteration metrics
# - Final structured Pydantic response
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import List, Optional, Any, Dict
from collections import Counter
import warnings

from langchain_core._api.deprecation import LangChainDeprecationWarning

warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


# ============================================================
# 1. Structured final response schema
# ============================================================

class AutonomousAgentResponse(BaseModel):
    """
    This is the final structured response returned by the agent.

    Important:
    - This does not replace result["messages"].
    - Tool-call metrics still come from result["messages"].
    - This is your validated final business output.
    """

    final_answer: str = Field(
        description="The final response to the user's request."
    )

    task_completed: bool = Field(
        description="Whether the agent completed the user's request."
    )

    reasoning_summary: str = Field(
        description="Brief user-facing summary of the steps the agent took. Do not include hidden chain-of-thought."
    )

    tools_used: List[str] = Field(
        description="Names of tools used by the agent during the run."
    )

    key_findings: List[str] = Field(
        description="Important facts, observations, or tool outputs used to support the final answer."
    )

    limitations: List[str] = Field(
        description="Any limitations, missing information, or uncertainty."
    )

    recommended_next_steps: List[str] = Field(
        description="Suggested next steps for the user, if any."
    )

    confidence: float = Field(
        description="Confidence score from 0.0 to 1.0."
    )


# ============================================================
# 2. Content helpers
# ============================================================

def normalize_content(content: Any) -> str:
    """
    Handles plain string content and list-of-blocks content.
    LangChain message content can be a string or structured blocks.
    """
    if content is None:
        return ""

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []

        for block in content:
            if isinstance(block, str):
                parts.append(block)

            elif isinstance(block, dict):
                if "text" in block:
                    parts.append(str(block["text"]))
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))

            else:
                parts.append(str(block))

        return "\n".join(parts)

    return str(content)


def get_final_answer_from_messages(result: dict) -> str:
    """
    Returns the final non-empty AIMessage content.

    Note:
    When using structured output, the final AIMessage.content may sometimes
    be empty or less useful depending on provider strategy. In that case,
    prefer result["structured_response"].final_answer for the final answer.
    """
    for msg in reversed(result.get("messages", [])):
        if isinstance(msg, AIMessage):
            text = normalize_content(msg.content).strip()
            if text:
                return text

    return ""


def get_structured_response(result: dict) -> Optional[AutonomousAgentResponse]:
    """
    Returns the final Pydantic structured response if LangChain produced one.
    """
    return result.get("structured_response")


def get_final_answer(result: dict) -> str:
    """
    Preferred final-answer accessor.

    Uses structured_response.final_answer when available.
    Falls back to the final AIMessage content.
    """
    structured = get_structured_response(result)

    if structured is not None and hasattr(structured, "final_answer"):
        return structured.final_answer

    return get_final_answer_from_messages(result)


# ============================================================
# 3. Metrics helpers
# ============================================================

def summarize_agent_metrics(result: dict) -> dict:
    """
    Summarize agent run metrics while preserving tool-call visibility.

    This function intentionally reads from result["messages"].
    Do not switch this to structured_response, or you will lose tool-call metrics.
    """
    messages = result.get("messages", [])

    ai_messages = [m for m in messages if isinstance(m, AIMessage)]
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    human_messages = [m for m in messages if isinstance(m, HumanMessage)]

    tool_calls = []

    token_totals = {
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
    }

    model_calls_with_usage = 0

    token_usage_by_step = []

    ai_step = 0

    for message_index, msg in enumerate(messages):
        if not isinstance(msg, AIMessage):
            continue

        ai_step += 1

        # ----------------------------------------------------
        # Tool calls requested by the model
        # ----------------------------------------------------
        msg_tool_calls = getattr(msg, "tool_calls", None) or []

        for call in msg_tool_calls:
            tool_calls.append({
                "name": call.get("name"),
                "args": call.get("args"),
                "id": call.get("id"),
                "message_index": message_index,
                "ai_step": ai_step,
            })

        # ----------------------------------------------------
        # Token usage
        # ----------------------------------------------------
        usage = getattr(msg, "usage_metadata", None)

        step_input_tokens = 0
        step_output_tokens = 0
        step_total_tokens = 0

        if usage:
            model_calls_with_usage += 1

            step_input_tokens = usage.get("input_tokens", 0) or 0
            step_output_tokens = usage.get("output_tokens", 0) or 0
            step_total_tokens = usage.get("total_tokens", 0) or 0

            token_totals["input_tokens"] += step_input_tokens
            token_totals["output_tokens"] += step_output_tokens
            token_totals["total_tokens"] += step_total_tokens

        token_usage_by_step.append({
            "message_index": message_index,
            "ai_step": ai_step,
            "input_tokens": step_input_tokens,
            "output_tokens": step_output_tokens,
            "total_tokens": step_total_tokens,
            "tool_call_count": len(msg_tool_calls),
            "has_tool_calls": len(msg_tool_calls) > 0,
        })

    tool_name_counts = Counter(call["name"] for call in tool_calls)

    structured = get_structured_response(result)

    if structured is not None:
        try:
            structured_response_dict = structured.model_dump()
        except Exception:
            structured_response_dict = str(structured)
    else:
        structured_response_dict = None

    return {
        # Final answer
        "final_answer": get_final_answer(result),
        "final_answer_from_messages": get_final_answer_from_messages(result),

        # Structured response
        "has_structured_response": structured is not None,
        "structured_response_type": type(structured).__name__ if structured is not None else None,
        "structured_response": structured,
        "structured_response_dict": structured_response_dict,

        # Message counts
        "message_count": len(messages),
        "human_message_count": len(human_messages),
        "ai_message_count": len(ai_messages),
        "tool_message_count": len(tool_messages),

        # Agent loop metrics
        "agent_iterations": len(ai_messages),

        # Tool metrics
        "tool_call_count": len(tool_calls),
        "tool_name_counts": dict(tool_name_counts),
        "tool_calls": tool_calls,
        "tool_call_sequence": [call["name"] for call in tool_calls],
        "unique_tools_used": len(tool_name_counts),

        # Token metrics
        "model_calls_with_usage": model_calls_with_usage,
        "token_usage_by_step": token_usage_by_step,
        **token_totals,
    }


def get_tool_outputs(result: dict) -> List[dict]:
    """
    Extract tool outputs from ToolMessage objects.
    """
    outputs = []

    for message_index, msg in enumerate(result.get("messages", [])):
        if isinstance(msg, ToolMessage):
            content = normalize_content(msg.content)

            outputs.append({
                "message_index": message_index,
                "tool_name": getattr(msg, "name", None),
                "tool_call_id": getattr(msg, "tool_call_id", None),
                "output": content,
                "output_chars": len(content),
                "is_empty": len(content.strip()) == 0,
            })

    return outputs


def print_agent_report(result: dict) -> None:
    """
    Print the same report style you already have, plus structured response.
    """
    metrics = summarize_agent_metrics(result)

    print("Final answer:")
    print(metrics["final_answer"])

    print("\nFinal answer from messages:")
    print(metrics["final_answer_from_messages"])

    print("\nRun metrics:")
    for k, v in metrics.items():
        if k not in [
            "final_answer",
            "final_answer_from_messages",
            "tool_calls",
            "structured_response",
            "structured_response_dict",
            "token_usage_by_step",
        ]:
            print(f"{k}: {v}")

    print("\nTool calls:")
    for call in metrics["tool_calls"]:
        print(f"- ai_step={call['ai_step']} | {call['name']}: {call['args']}")

    print("\nToken usage by step:")
    for step in metrics["token_usage_by_step"]:
        print(
            f"- ai_step={step['ai_step']} | "
            f"input={step['input_tokens']} | "
            f"output={step['output_tokens']} | "
            f"total={step['total_tokens']} | "
            f"tool_calls={step['tool_call_count']}"
        )

    print("\nTool outputs:")
    for output in get_tool_outputs(result):
        print(
            f"- message_index={output['message_index']} | "
            f"tool_name={output['tool_name']} | "
            f"chars={output['output_chars']} | "
            f"preview={output['output'][:250]}"
        )

    print("\nStructured response:")
    structured = metrics["structured_response"]

    if structured is None:
        print("No structured_response found.")
    else:
        print(structured)

        print("\nStructured response JSON:")
        print(structured.model_dump_json(indent=2))


# ============================================================
# 4. Tool definitions
# ============================================================

@tool
def run_sql(query: str) -> str:
    """
    Run a read-only SQL query and return results.

    Demo placeholder:
    Replace this with your actual SQL execution logic.
    """
    return f"SQL tool received query: {query}"


@tool
def validate_answer(answer: str, evidence: str) -> str:
    """
    Critique the answer for missing evidence, bad assumptions, or calculation errors.
    """
    if not answer.strip():
        return "Validation failed: answer is empty."

    if not evidence.strip():
        return "Validation warning: evidence is empty or limited."

    return "Validation passed: the answer is supported by the provided evidence."


@tool
def search_docs(query: str) -> str:
    """
    Search internal documents for relevant context.

    Demo placeholder:
    Replace this with your vector DB / RAG search.
    """
    return f"No internal documents found for query: {query}"


@tool
def save_artifact(content: str) -> str:
    """
    Save the final output to a file or database.

    Demo placeholder:
    Replace this with actual file/database persistence.
    """
    return f"Artifact save simulated. Content length: {len(content)} characters."


@tool
def weather(location: str) -> str:
    """
    Get the current weather for a location.
    """
    normalized = location.lower().strip()

    if normalized == "new york":
        return "The current weather in New York is sunny, 75°F."

    if normalized == "delano":
        return "The current weather in Delano is foggy, 60°F."

    return f"Sorry, I don't have weather data for {location}."


@tool
def web_search(query: str) -> str:
    """
    Search the web for information.

    Demo placeholder:
    Replace this with Tavily, SerpAPI, Google Custom Search, Exa, etc.
    """
    normalized = query.lower().strip()

    if normalized == "latest on ai":
        return "The latest news on AI is about OpenCode, an alternative to Claude Code, and it is free."

    return f"Sorry, I don't have web search capabilities for the query: {query}"


# ============================================================
# 5. LLM setup
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,
    project="zen-general-377713",
    location="global",
)


# ============================================================
# 6. Agent setup with structured response
# ============================================================

agent = create_agent(
    model=llm,
    tools=[
        run_sql,
        validate_answer,
        search_docs,
        save_artifact,
        weather,
        web_search,
    ],
    response_format=AutonomousAgentResponse,
    system_prompt="""
    You are an autonomous agent.

    You may use tools repeatedly when needed.

    Before finalizing:
    - Make sure the answer is grounded in tool results when tools were used.
    - Do not fabricate tool results.
    - Stop when the answer is complete and verified.

    Your final answer must be returned using the required structured response schema.

    In the structured response:
    - final_answer should directly answer the user's request.
    - tools_used should list the actual tools used.
    - key_findings should summarize important facts from tool outputs.
    - limitations should mention missing information or uncertainty.
    - confidence should be between 0.0 and 1.0.
    """
)


# ============================================================
# 7. Invoke the agent
# ============================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": 'Search the web for "latest on ai" if it is hotter than 50 degrees in Delano.',
            }
        ]
    },
    config={
        # Optional, but useful for autonomous agents.
        # This is not exactly "max tool calls"; it is the graph recursion cap.
        "recursion_limit": 50
    },
)


# ============================================================
# 8. Inspect raw messages if desired
# ============================================================

print("Raw messages:")
for msg in result["messages"]:
    print(msg)
    print("-" * 100)


# ============================================================
# 9. Print full report
# ============================================================

print_agent_report(result)


# ============================================================
# 10. Programmatic access
# ============================================================

metrics = summarize_agent_metrics(result)
structured = get_structured_response(result)

# Tool-call metrics still available:
tool_call_count = metrics["tool_call_count"]
tool_name_counts = metrics["tool_name_counts"]
tool_calls = metrics["tool_calls"]
agent_iterations = metrics["agent_iterations"]
input_tokens = metrics["input_tokens"]
output_tokens = metrics["output_tokens"]
total_tokens = metrics["total_tokens"]

# Structured final response also available:
if structured is not None:
    final_structured_answer = structured.final_answer
    structured_dict = structured.model_dump()
else:
    final_structured_answer = None
    structured_dict = None

print("\nProgrammatic values:")
print("tool_call_count:", tool_call_count)
print("tool_name_counts:", tool_name_counts)
print("agent_iterations:", agent_iterations)
print("input_tokens:", input_tokens)
print("output_tokens:", output_tokens)
print("total_tokens:", total_tokens)
print("final_structured_answer:", final_structured_answer)
print("structured_dict:", structured_dict)

Raw messages:
content='Search the web for "latest on ai" if it is hotter than 50 degrees in Delano.' additional_kwargs={} response_metadata={} id='ba36d8d0-3f80-4524-991a-6c52d670c090'
----------------------------------------------------------------------------------------------------
content=[] additional_kwargs={'function_call': {'name': 'weather', 'arguments': '{"location": "Delano"}'}, '__gemini_function_call_thought_signatures__': {'dfff914c-46fa-40af-a629-d96092618404': 'AY89a1+JgkBNLYi80NMfGmUcJfzku5f8kutwp1CBmaWRXPkJ62UlwxIXS8+vvJGKigd/twMK+yloROzS23RfChlvxBqh0EezBn2RC71Jwy5swcogDCvSB0Cnvx1EFOlv62zty973qb59xu5/lyNT9NDaKAn2siO+/K9umagfAmviqHujKmJJNw2XzG+6aPhqABKoaYazexQPZ8sDz5sTalkji07TTyZp05Nvnfkn9D1kCSUfY3lielMdBZpIkUyg1vLLzqxYK1DbO2pW4yG1UJFpXlVk9u9MAgGRBwg8p5DxMn8EDDRiV7PKCm8syVDk05r4lel5WPnsC8XwjrLZLufZbRZp6KJm5uUlYsUhs21LrrC+BfFk0bbjfnTC0ZFWLA04JoG05SKaltFuRmIXBNFMN79f+Se9t8K08flwK5BeBadSJHBcope8yvdBT2qXWBAO+iMkaYBCo+8ay2QjCUYq7FnWqJVSsQ=='}} response_metadata={'finish_rea